# TP 6 — DNS : le réseau répond, mais le nom ne résout pas

**Public cible :** étudiant, technicien, NetOps ou cyber débutant. **Durée :
60 minutes.** Tout est synthétique et hors ligne : aucun résolveur public,
socket, fichier, privilège ou paquet émis.

> Mission : suivre une résolution de bout en bout, expliquer le cache et
> distinguer NXDOMAIN, NODATA, SERVFAIL et valeur périmée avant de proposer la
> prochaine mesure. Une **réponse DNS ≠ connectivité applicative**.


## Contrat d'apprentissage

Vous relierez cinq rôles sans les confondre : stub client, **résolveur récursif**,
racine, serveur de TLD et serveur **autoritatif**. Vous observerez :

- la **délégation** lors d'un cache froid ;
- le **TTL** et la différence cache chaud / valeur périmée ;
- le **cache négatif** d'un NXDOMAIN ;
- **NODATA** : le nom existe, mais pas le type demandé ;
- **SERVFAIL** : le résolveur ne peut pas fournir de réponse utilisable.

Avant chaque scénario, prédisez le code, la source de la réponse et le prochain
acteur interrogé. Les limites du modèle sont rappelées en fin de TP.


In [ ]:
import base64
import hashlib
import html
import ipaddress
import json
import struct

SEED = 73
TYPE_CODES = {"A": 1, "AAAA": 28}

def encode_name(name):
    labels = name.rstrip(".").split(".")
    return b"".join(bytes((len(label),)) + label.encode("ascii") for label in labels) + b"\x00"

def decode_name(payload, offset):
    labels = []
    while True:
        length = payload[offset]
        offset += 1
        if length == 0:
            return ".".join(labels), offset
        if length & 0xC0:
            raise ValueError("compression intentionally unsupported in teaching query parser")
        labels.append(payload[offset:offset + length].decode("ascii"))
        offset += length

def build_query(name, qtype="A", transaction_id=None):
    txid = transaction_id if transaction_id is not None else (0x7000 + SEED)
    header = struct.pack("!HHHHHH", txid, 0x0100, 1, 0, 0, 0)
    return header + encode_name(name) + struct.pack("!HH", TYPE_CODES[qtype], 1)

def parse_query(payload):
    txid, flags, qdcount, ancount, nscount, arcount = struct.unpack("!HHHHHH", payload[:12])
    name, offset = decode_name(payload, 12)
    qtype, qclass = struct.unpack("!HH", payload[offset:offset + 4])
    return {"txid": txid, "flags": flags, "qdcount": qdcount, "name": name, "qtype": qtype, "qclass": qclass}

def build_a_response(query, address, ttl=30, rcode=0):
    parsed = parse_query(query)
    flags = 0x8180 | (rcode & 0xF)
    answer_count = 1 if rcode == 0 else 0
    header = struct.pack("!HHHHHH", parsed["txid"], flags, 1, answer_count, 0, 0)
    question = query[12:]
    if answer_count == 0:
        return header + question
    answer = b"\xc0\x0c" + struct.pack("!HHIH", 1, 1, ttl, 4) + ipaddress.IPv4Address(address).packed
    return header + question + answer

def parse_header(payload):
    txid, flags, qd, an, ns, ar = struct.unpack("!HHHHHH", payload[:12])
    return {"txid": txid, "qr": (flags >> 15) & 1, "rcode": flags & 0xF, "questions": qd, "answers": an}

dns_query_bytes = build_query("www.lab.test", "A")
dns_response_bytes = build_a_response(dns_query_bytes, "192.0.2.40", ttl=30)
parsed_query = parse_query(dns_query_bytes)
parsed_response = parse_header(dns_response_bytes)

assert parsed_query["name"] == "www.lab.test" and parsed_query["qtype"] == 1
assert parsed_response == {"txid": 0x7000 + SEED, "qr": 1, "rcode": 0, "questions": 1, "answers": 1}
print("Message DNS réel en mémoire :", len(dns_query_bytes), "octets requête ;", len(dns_response_bytes), "octets réponse")
print("Requête hex :", dns_query_bytes.hex(" ")[:95], "…")


## 1. Prédire six scénarios

| Scénario | Question | Prédiction à écrire avant exécution |
|---|---|---|
| froid | `www.lab.test A` | délégations puis CNAME et A |
| chaud | même question à t=5 | réponse du cache récursif |
| périmé | autoritatif modifié à t=15 | ancienne valeur jusqu'à expiration |
| négatif | nom créé après un NXDOMAIN | NXDOMAIN conservé jusqu'au TTL négatif |
| NODATA | `api.lab.test AAAA` | NOERROR, aucune AAAA |
| SERVFAIL | autoritatif indisponible | échec sans réponse de donnée |

Ne déduisez jamais la disponibilité HTTP d'une seule réponse A.


In [ ]:
NEGATIVE_TTL = 20
AUTHORITATIVE_CHANGE_AT = 15
NEW_NAME_CREATED_AT = 10
dns_trace = []

def authoritative_answer(name, qtype, now, available=True):
    if not available or name == "broken.lab.test":
        return {"status": "SERVFAIL", "answers": [], "ttl": 0}
    if name == "www.lab.test" and qtype == "A":
        address = "192.0.2.41" if now >= AUTHORITATIVE_CHANGE_AT else "192.0.2.40"
        return {
            "status": "NOERROR",
            "answers": [("www.lab.test", "CNAME", "api.lab.test", 60), ("api.lab.test", "A", address, 30)],
            "ttl": 30,
        }
    if name == "api.lab.test" and qtype == "A":
        address = "192.0.2.41" if now >= AUTHORITATIVE_CHANGE_AT else "192.0.2.40"
        return {"status": "NOERROR", "answers": [(name, "A", address, 30)], "ttl": 30}
    if name == "api.lab.test" and qtype == "AAAA":
        return {"status": "NODATA", "answers": [], "ttl": NEGATIVE_TTL}
    if name == "new.lab.test" and now >= NEW_NAME_CREATED_AT and qtype == "A":
        return {"status": "NOERROR", "answers": [(name, "A", "192.0.2.55", 30)], "ttl": 30}
    return {"status": "NXDOMAIN", "answers": [], "ttl": NEGATIVE_TTL}

def record_event(scenario, step, source, target, message, status="IN_PROGRESS"):
    dns_trace.append({
        "scenario": scenario, "step_ms": step, "source": source, "target": target,
        "message": message, "status": status,
    })

def resolve(cache, scenario, name, qtype, now, authoritative_available=True):
    key = (name, qtype)
    record_event(scenario, 0, "Stub", "Récursif", f"{name} {qtype}")
    cached = cache.get(key)
    if cached and now < cached["expires_at"]:
        result = dict(cached["result"])
        result["source"] = "cache"
        record_event(scenario, 2, "Récursif", "Stub", f"cache hit · {result['status']}", result["status"])
        return result
    record_event(scenario, 2, "Récursif", "Racine", "où est .test ?")
    record_event(scenario, 5, "Racine", "Récursif", "délégation .test")
    record_event(scenario, 7, "Récursif", "TLD .test", "où est lab.test ?")
    record_event(scenario, 10, "TLD .test", "Récursif", "délégation lab.test")
    record_event(scenario, 12, "Récursif", "Autoritatif", f"{name} {qtype}")
    result = authoritative_answer(name, qtype, now, authoritative_available)
    record_event(scenario, 17, "Autoritatif", "Récursif", result["status"], result["status"])
    if result["status"] != "SERVFAIL":
        cache[key] = {"expires_at": now + result["ttl"], "result": dict(result)}
    result = dict(result)
    result["source"] = "autoritatif"
    record_event(scenario, 20, "Récursif", "Stub", result["status"], result["status"])
    return result


In [ ]:
main_cache = {}
negative_cache = {}
diagnostic_cache = {}

RESULTS = {
    "cold": resolve(main_cache, "froid", "www.lab.test", "A", now=0),
    "warm": resolve(main_cache, "chaud", "www.lab.test", "A", now=5),
    "stale": resolve(main_cache, "périmé", "www.lab.test", "A", now=20),
    "refreshed": resolve(main_cache, "rafraîchi", "www.lab.test", "A", now=31),
    "nxdomain": resolve(negative_cache, "NXDOMAIN initial", "new.lab.test", "A", now=0),
    "negative_cached": resolve(negative_cache, "cache négatif", "new.lab.test", "A", now=12),
    "negative_expired": resolve(negative_cache, "négatif expiré", "new.lab.test", "A", now=21),
    "nodata": resolve(diagnostic_cache, "NODATA", "api.lab.test", "AAAA", now=0),
    "servfail": resolve(diagnostic_cache, "SERVFAIL", "broken.lab.test", "A", now=0, authoritative_available=False),
}

def first_address(result):
    values = [row[2] for row in result["answers"] if row[1] == "A"]
    return values[0] if values else "—"

assert RESULTS["warm"]["source"] == "cache"
assert first_address(RESULTS["stale"]) == "192.0.2.40"
assert first_address(RESULTS["refreshed"]) == "192.0.2.41"
assert RESULTS["negative_cached"]["status"] == "NXDOMAIN"
assert first_address(RESULTS["negative_expired"]) == "192.0.2.55"
assert RESULTS["nodata"]["status"] == "NODATA"
assert RESULTS["servfail"]["status"] == "SERVFAIL"

for key in ("cold", "warm", "stale", "refreshed", "nxdomain", "negative_cached", "negative_expired", "nodata", "servfail"):
    result = RESULTS[key]
    print(f"{key:17s} {result['status']:8s} source={result['source']:11s} valeur={first_address(result)}")


## 2. Diagnostiquer sans confondre les codes

- **NXDOMAIN** : le nom est déclaré inexistant ; vérifier le cache négatif et la zone.
- **NODATA** : le nom existe, mais aucun enregistrement du type demandé.
- **SERVFAIL** : la chaîne n'a pas produit de réponse utilisable ; chercher
  disponibilité autoritative, délégation, validation ou timeout.
- **NOERROR + adresse** : la résolution a réussi ; cela ne prouve ni route, ni
  port ouvert, ni TLS, ni santé de l'application.

Demandez toujours : qui a répondu, depuis quel cache, avec quel TTL, et quelle
observation falsifierait votre diagnostic ?


In [ ]:
ACTORS = ("Stub", "Récursif", "Racine", "TLD .test", "Autoritatif")
X = {actor: 100 + index * 230 for index, actor in enumerate(ACTORS)}
STATUS_COLORS = {"NOERROR": "#22c55e", "NXDOMAIN": "#ef4444", "NODATA": "#f59e0b", "SERVFAIL": "#a855f7", "IN_PROGRESS": "#38bdf8"}

class RichHTML:
    def __init__(self, markup):
        self.markup = markup

    def _repr_html_(self):
        return self.markup

def dns_dashboard():
    events = [event for event in dns_trace if event["scenario"] == "froid"]
    parts = [
        '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1200 760" role="img" aria-labelledby="dns-title dns-desc">',
        '<title id="dns-title">Waterfall DNS et chronologie des caches</title>',
        '<desc id="dns-desc">Résolution froide, cache positif et négatif, codes de réponse.</desc>',
        '<rect width="1200" height="760" rx="28" fill="#07111f"/>',
        '<text x="50" y="58" fill="#f8fafc" font-size="31" font-family="sans-serif" font-weight="700">DNS — qui répond, avec quelle fraîcheur ?</text>',
        '<text x="50" y="88" fill="#94a3b8" font-size="16" font-family="sans-serif">délégation · récursion · TTL · cache négatif · code ≠ cause complète</text>',
    ]
    for actor in ACTORS:
        x = X[actor]
        parts.append(f'<text x="{x}" y="130" text-anchor="middle" fill="#e2e8f0" font-size="17" font-family="sans-serif" font-weight="700">{html.escape(actor)}</text>')
        parts.append(f'<line x1="{x}" y1="145" x2="{x}" y2="455" stroke="#334155" stroke-dasharray="6 7"/>')
    for index, event in enumerate(events):
        y = 170 + index * 38
        x1, x2 = X[event["source"]], X[event["target"]]
        color = STATUS_COLORS.get(event["status"], "#38bdf8")
        direction = 1 if x2 >= x1 else -1
        parts.append(f'<line x1="{x1}" y1="{y}" x2="{x2 - direction * 10}" y2="{y}" stroke="{color}" stroke-width="3"/>')
        parts.append(f'<polygon points="{x2},{y} {x2 - direction * 12},{y - 6} {x2 - direction * 12},{y + 6}" fill="{color}"/>')
        parts.append(f'<text x="{(x1 + x2) / 2:.1f}" y="{y - 7}" text-anchor="middle" fill="#cbd5e1" font-size="12" font-family="sans-serif">{html.escape(event["message"])}</text>')
    parts.extend([
        '<rect x="50" y="485" width="1100" height="105" rx="18" fill="#102033"/>',
        '<text x="75" y="520" fill="#f8fafc" font-size="20" font-family="sans-serif" font-weight="700">Timeline valeur A</text>',
        '<line x1="270" y1="545" x2="1090" y2="545" stroke="#475569" stroke-width="4"/>',
    ])
    timeline = ((0, "cache .40", "#38bdf8"), (15, "autoritatif → .41", "#f97316"), (20, "cache encore .40", "#ef4444"), (30, "TTL expire", "#f59e0b"), (31, "cache .41", "#22c55e"))
    for when, label, color in timeline:
        x = 270 + when / 31 * 820
        parts.append(f'<circle cx="{x:.1f}" cy="545" r="8" fill="{color}"/>')
        label_x = x - 8 if when == 30 else x + 8 if when == 31 else x
        label_y = 570 if when not in (20, 31) else 586
        anchor = "end" if when == 30 else "start" if when == 31 else "middle"
        parts.append(f'<text x="{label_x:.1f}" y="{label_y}" text-anchor="{anchor}" fill="#cbd5e1" font-size="12" font-family="sans-serif">t={when} {html.escape(label)}</text>')
    cards = (("NXDOMAIN", "nom absent + cache négatif"), ("NODATA", "nom présent, type absent"), ("SERVFAIL", "aucune réponse utilisable"), ("NOERROR", "DNS OK, application inconnue"))
    for index, (status, label) in enumerate(cards):
        x = 50 + index * 277
        color = STATUS_COLORS[status]
        parts.append(f'<rect x="{x}" y="620" width="255" height="95" rx="16" fill="#102033" stroke="{color}" stroke-width="2"/>')
        parts.append(f'<text x="{x + 18}" y="655" fill="{color}" font-size="19" font-family="sans-serif" font-weight="700">{status}</text>')
        parts.append(f'<text x="{x + 18}" y="684" fill="#cbd5e1" font-size="13" font-family="sans-serif">{html.escape(label)}</text>')
    parts.append('</svg>')
    return "".join(parts)

DNS_DASHBOARD = dns_dashboard()
assert "<svg" in DNS_DASHBOARD and DNS_DASHBOARD.count("NXDOMAIN") >= 1
RichHTML(DNS_DASHBOARD)


## 3. Transfert : choisir la prochaine preuve

Pour chaque ticket, proposez une seule prochaine mesure : vider un cache de
laboratoire, interroger directement l'autoritatif, vérifier la délégation,
comparer le TTL restant, demander un autre type, ou tester séparément la
connectivité applicative. L'action doit être autorisée, réversible et journalisée.

Ne contournez jamais une politique DNS et n'envoyez aucune requête vers une
infrastructure qui n'appartient pas au banc.


In [ ]:
report_rows = []
for key in ("cold", "warm", "stale", "refreshed", "nxdomain", "negative_cached", "negative_expired", "nodata", "servfail"):
    result = RESULTS[key]
    report_rows.append(
        f"- {key}: status={result['status']}; source={result['source']}; valeur={first_address(result)}"
    )

incident_report = '''# Rapport DNS synthétique

## Question
Pourquoi le nom ou le service semble-t-il indisponible ?

## Observations
''' + "\n".join(report_rows) + '''

## Conclusion
Les codes et caches distinguent plusieurs mécanismes. Une réponse DNS n'établit pas la connectivité applicative.

## Prochaine preuve
Répéter sur un banc autorisé avec trace horodatée, TTL restant, identité du répondant et test applicatif séparé.

## Frontière
Résolveur pédagogique hors ligne ; aucune requête réelle ni conformité DNS/DNSSEC revendiquée.
'''

trace_json = json.dumps(dns_trace, ensure_ascii=False, sort_keys=True, indent=2)
report_digest = hashlib.sha256(incident_report.encode("utf-8")).hexdigest()
trace_uri = base64.b64encode(trace_json.encode("utf-8")).decode("ascii")
report_uri = base64.b64encode(incident_report.encode("utf-8")).decode("ascii")

download_html = f'''
<div style="font-family:system-ui;padding:18px;border:1px solid #38bdf8;border-radius:14px;background:#effaff">
  <h3 style="margin-top:0">Dossier DNS reproductible</h3>
  <p>{len(dns_trace)} événements · rapport SHA-256 <code>{report_digest[:16]}</code></p>
  <p><a download="dns_trace.json" href="data:application/json;base64,{trace_uri}">⬇ Télécharger la trace DNS</a><br>
  <a download="rapport_incident_dns.md" href="data:text/markdown;base64,{report_uri}">⬇ Télécharger le rapport</a></p>
</div>
'''
assert len(dns_trace) >= 40 and "SERVFAIL" in incident_report
RichHTML(download_html)


## 4. Limites de preuve

- zones, horloges, délégations et pannes entièrement synthétiques ;
- format binaire limité à une requête A et une réponse A sans compression générale ;
- pas d'EDNS, TCP fallback, DoT, DoH, anycast, récursion concurrente ou lame duck ;
- pas de validation cryptographique DNSSEC ni de mesure de résolveur réel ;
- le modèle de cache simplifie RFC et politiques produits ;
- aucun résultat ne prouve la disponibilité réseau ou applicative.

Le TP enseigne une démarche causale et du vocabulaire observable, pas une
implémentation de production ni une autorisation d'interroger Internet.
